In [1]:
import os
from dotenv import load_dotenv
import xarray as xr

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import date

import analysis_utils
import isku_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'isku_utils' from '/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py'>

In [39]:
load_dotenv()
DATA_DIR = os.environ["DATA_DIR"]
# Baseline period Monthly Climatology
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")
# Define Forecast Months
FC_MONTHS = [9, 10, 11, 12, 1, 2]
FC_PERIOD = slice("2026-09-01", "2027-02-28")

config = analysis_utils.ImpactConfig( version = "v260910",
                                     baseline_period=BASELINE_PERIOD, 
                                     rate=False, 
                                     months = FC_MONTHS,
                                     hotonly = "hotonly", 
                                     dims = ['number', 'sample'])

# Define Forecast
EFFECTS_URI = "/home/emily_zuetell/projects/poreallas/data/v20260909_effects_with_betas.zarr"

In [40]:
# Projection Effects
effect = xr.open_datatree(os.path.join(DATA_DIR, EFFECTS_URI), consolidated=False)

In [ ]:
analysis_utils.make_csv(effect, config, group_level="IR")

In [41]:
analysis_utils.make_csv(effect, config, group_level="ADM1")

source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped
source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped
source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped
source coverage: 6 unmatched source units (first 6: [['BRA

KeyError: "['region', 'ISO'] not in index"

In [33]:
impact = config.compute_impact(effect, ensemble=False).mean(dim = 'sample')

In [5]:
import cil_regionalization as cilreg
from cil_regionalization.config import SourceUnitPolicies

DATA_VERSION = "world-combo-201710"  # the impact region version of the sample

In [34]:
weights = cilreg.fetch_weights(
            "gadm41-adm1-per-destination" if config.rate else "gadm41-adm1-per-source"
        )
impact = analysis_utils.redistribute_adm1(impact, config, weights)

source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped


In [26]:
rate_l = "rate" if config.rate else "total"
baseline_tag = analysis_utils._baseline_tag(config.baseline_period)

impact = config.compute_impact(effect.chunk({dim: -1 for dim in config.dims}), ensemble=True).sel(number = [1, 2, 3])
impact = impact.sel(month=config.months)

polygon = config.polygons
# Aggregate Impact Regions to group_level
impact, merge_key, base_cols = analysis_utils.aggregate_impact(impact, config, 'ADM1')

stat_cols = ["median", "p17", "p83", "likely_range_IPCC", "mean", "std", "min", "max", "p10", "p90"]

source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped


In [27]:
impact

<xarray.DataArray 'value' (GID_1: 3683, number: 3, sample: 15, month: 6)> Size: 8MB
array([[[[ 0.00000000e+00,  0.00000000e+00,  7.84831900e-02,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  2.76148448e-02,
          -4.79443553e-04,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00, -1.08108831e-01,
           4.33410387e-02, -2.13663636e-04,  0.00000000e+00],
         ...,
         [ 0.00000000e+00,  0.00000000e+00,  2.06774396e-02,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00, -7.79258478e-01,
           6.34513616e-01,  7.22071562e-04,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00, -6.03263755e-03,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00]],

        [[ 0.00000000e+00,  0.00000000e+00,  2.05925392e-01,
           0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  0.00000000e+00,  2.99474811e-01,
          -4.79443553e-04,  0.00000000e+00,  0.00000000e+00],
         [ 0.00000000e+00,  1.04994467e-03,  6.11450037e-01,
           1.91466486e-02,  1.05078835e-03,  0.00000000e+00],
...
          -3.48576954e+00, -1.63000855e+00,  1.46795059e+01],
         [ 8.17534416e+01,  7.51885547e+01, -8.99302707e+00,
          -1.47131670e+01, -2.72464000e+00,  4.36116128e+01],
         [ 3.42648114e+01,  3.12152107e+01, -3.44203540e+00,
          -5.96989975e+00, -9.62247273e-01,  1.78577087e+01]],

        [[ 3.66070905e+00,  3.15368557e+00, -1.53967017e+00,
           1.72539492e+00,  1.25860785e+01,  1.95503719e+01],
         [ 1.47412919e+01,  1.42993577e+01, -7.56492102e+00,
           6.23789193e+00,  3.93171804e+01,  5.77777427e+01],
         [ 1.34619267e+01,  1.36004065e+01, -8.42359813e+00,
           5.80327193e+00,  3.26411884e+01,  4.85200756e+01],
         ...,
         [ 4.52658099e+00,  3.98201670e+00, -1.81249710e+00,
           2.11003970e+00,  1.49975560e+01,  2.21729161e+01],
         [ 1.59167007e+01,  1.60002872e+01, -1.04027133e+01,
           6.98746788e+00,  3.95260494e+01,  6.05363351e+01],
         [ 6.87812485e+00,  6.90791481e+00, -3.95272535e+00,
           3.02334934e+00,  1.69206807e+01,  2.47095617e+01]]]],
      shape=(3683, 3, 15, 6))
Coordinates:
  * GID_1    (GID_1) object 29kB '?' 'ABW' 'AFG.10_1' ... 'ZWE.8_1' 'ZWE.9_1'
    GID_0    (GID_1) str 41kB ...
  * number   (number) int64 24B 1 2 3
  * sample   (sample) int64 120B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
  * month    (month) int64 48B 1 2 9 10 11 12

In [28]:
_polygons_impact = analysis_utils.dataset_to_dataframe(analysis_utils.compute_stats(impact, dim=config.dims))
wide = _polygons_impact.pivot(
    index=base_cols,
    columns="month", values=stat_cols,
)

In [30]:
wide.reset_index()

GID_0     GID_1     median                                         \
month                          1             2          9            10   
0       ABW       ABW   0.000002  0.000000e+00  -0.212417  1.464238e-06   
1       AFG  AFG.10_1   0.000000  0.000000e+00  -0.002267 -1.994652e-09   
2       AFG  AFG.11_1   0.000000  0.000000e+00  11.315119 -7.146662e-01   
3       AFG  AFG.12_1   0.000000 -9.235204e-08  -4.972470  5.182776e-01   
4       AFG  AFG.13_1   0.000000 -1.525151e-06  -2.662206 -6.420517e-04   
...     ...       ...        ...           ...        ...           ...   
3678    ZWE   ZWE.5_1  17.959054  1.406675e+01  -3.449896 -2.803206e+00   
3679    ZWE   ZWE.6_1  21.413226  1.925313e+01  -6.876679 -8.618902e+00   
3680    ZWE   ZWE.7_1  34.866845  1.572934e+01  -7.890598 -6.402815e+00   
3681    ZWE   ZWE.8_1  30.197386  1.545365e+01  -6.343368 -8.380992e+00   
3682    ZWE   ZWE.9_1  27.523912  6.319854e+00  -3.290894 -2.899627e+00   

                                         p17                ...        p10  \
month            11         12             1             2  ...          9   
0      7.321192e-07   0.000002  0.000000e+00 -1.464238e-06  ...  -0.897303   
1     -1.280171e-10   0.000000 -3.362856e-11 -6.555406e-11  ...  -0.189569   
2     -1.725354e-02   0.000000 -2.708784e-02 -1.182578e-03  ... -10.026409   
3     -1.272961e-03   0.000000 -6.031723e-02 -8.724993e-03  ... -22.845215   
4     -3.567818e-04   0.000000 -6.348473e-03 -8.443879e-02  ... -10.387197   
...             ...        ...           ...           ...  ...        ...   
3678  -1.971994e+00  13.582958  6.817555e+00  3.640554e+00  ... -11.899560   
3679  -7.893438e+00  25.122191  7.391144e+00  6.777689e+00  ... -21.250525   
3680  -3.658367e-01  35.083812  1.180302e+01  4.496950e+00  ... -20.903532   
3681  -4.815892e+00  34.573691  1.067645e+01  6.976966e+00  ... -16.634518   
3682  -1.192360e+00  27.350183  6.113786e+00  1.149062e+00  ...  -9.823218   

                                              p90                        \
month         10         11         12          1          2          9   
0       0.000000  -0.000005   0.000000   1.475893   1.235351   0.000000   
1      -0.001892  -0.086932  -0.004845   0.000000   0.000000   0.041220   
2      -4.951392  -2.661657  -0.030799   0.000000   0.022584  48.642509   
3      -2.301849  -2.030698  -0.048838   0.000000   0.000000   9.192653   
4      -0.343558  -1.044672  -0.056343   0.000000   0.000000   7.916757   
...          ...        ...        ...        ...        ...        ...   
3678  -12.213920 -22.860033   1.415769  45.103359  49.891272  -0.753849   
3679  -25.992405 -22.390913  -0.613044  61.825211  56.592960  -2.395544   
3680  -22.407903 -16.631064  19.370692  98.806169  86.812415  -1.804640   
3681  -24.886560 -11.123523   5.268457  81.778173  40.860008  -2.676600   
3682  -10.596035  -2.851924  12.919861  78.475101  38.476481  -0.766661   

                                              
month         10            11            12  
0       0.321631  9.173887e-01  1.092125e+00  
1       0.001847 -6.529777e-14  0.000000e+00  
2      57.942662 -7.195891e-11  0.000000e+00  
3      15.265148 -3.840492e-08  3.210438e-05  
4       2.569174  0.000000e+00  2.107099e-11  
...          ...           ...           ...  
3678   25.882298  4.402532e+01  6.321025e+01  
3679   45.410322  7.454749e+01  8.569223e+01  
3680    7.121560  4.447523e+01  7.973536e+01  
3681   24.189440  7.471274e+01  8.580602e+01  
3682    3.958990  2.051368e+01  5.943290e+01  

[3683 rows x 62 columns]

In [31]:


wide.columns = [f"month {m} {stat}" for stat, m in wide.columns]
stat_col_names = wide.columns.difference(base_cols)
wide[stat_col_names] = wide[stat_col_names].round(0).astype("Int64")
wide = wide.reset_index()


In [32]:
wide

,GID_0,GID_1,month 1 median,month 2 median,month 9 median,month 10 median,month 11 median,month 12 median,month 1 p17,month 2 p17,...,month 9 p10,month 10 p10,month 11 p10,month 12 p10,month 1 p90,month 2 p90,month 9 p90,month 10 p90,month 11 p90,month 12 p90
0,ABW,ABW,0,0,0,0,0,0,0,0,...,-1,0,0,0,1,1,0,0,1,1
1,AFG,AFG.10_1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,AFG,AFG.11_1,0,0,11,-1,0,0,0,0,...,-10,-5,-3,0,0,0,49,58,0,0
3,AFG,AFG.12_1,0,0,-5,1,0,0,0,0,...,-23,-2,-2,0,0,0,9,15,0,0
4,AFG,AFG.13_1,0,0,-3,0,0,0,0,0,...,-10,0,-1,0,0,0,8,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3678,ZWE,ZWE.5_1,18,14,-3,-3,-2,14,7,4,...,-12,-12,-23,1,45,50,-1,26,44,63
3679,ZWE,ZWE.6_1,21,19,-7,-9,-8,25,7,7,...,-21,-26,-22,-1,62,57,-2,45,75,86
3680,ZWE,ZWE.7_1,35,16,-8,-6,0,35,12,4,...,-21,-22,-17,19,99,87,-2,7,44,80
3681,ZWE,ZWE.8_1,30,15,-6,-8,-5,35,11,7,...,-17,-25,-11,5,82,41,-3,24,75,86
